# Travelmind API (AI Trip Planner)

In [1]:
#importing libraries
import os
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet


In [2]:
load_dotenv()
client=OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
def create_pdf(text):
    path="Travel_Itinerary.pdf"
    doc=SimpleDocTemplate(path)
    styles=getSampleStyleSheet()
    story=[Paragraph(line or " ",styles["BodyText"]) for line in text.split("\n")]
    doc.build(story)
    return path

In [4]:
def plan(dest,days,budget,traveler,interests):
    prompt=f"""Create a {days}-day travel itinerary.
Destination: {dest}
Budget: {budget}
Traveler: {traveler}
Interests: {", ".join(interests)}
Include daily plan, hotels, food, transport, packing list and budget."""
    resp=client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role":"system","content":"You are an expert travel planner."},
            {"role":"user","content":prompt}
        ]
    )
    text=resp.choices[0].message.content
    pdf=create_pdf(text)
    return text,pdf


In [5]:

with gr.Blocks(title="AI Travel Planner") as demo:
    gr.Markdown("# 🌍 TraveMind AI Travel Planner")
    d=gr.Textbox(label="Destination")
    days=gr.Slider(1,15,value=5,label="Days")
    budget=gr.Radio(["Budget","Medium","Luxury"],value="Medium",label="Budget")
    traveler=gr.Dropdown(["Solo","Couple","Family","Friends"],value="Solo",label="Traveler")
    interests=gr.CheckboxGroup(["Adventure","Food","Nature","Shopping","History","Beaches"],value=["Food"])
    out=gr.Markdown()
    pdf=gr.File(label="PDF")
    gr.Button("Generate").click(plan,[d,days,budget,traveler,interests],[out,pdf])

demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


C:\Users\HP\anaconda new\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
